<a href="https://colab.research.google.com/github/ithelga/beeline-banner-ab-test/blob/dev/notebooks/Team1_HW3_Hypotheses.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SmartAds Efficiency**: Оптимизация эффективности маркетинговых каналов

## [STAGE 2] **Формулирование гипотез**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
import polars as pl
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

colors = ["#FFAFCC", "#FFC8DD", "#CDB4DB", "#BDE0FE", "#A2D2FF"] # СТАРЫЕ ЦВЕТА!!!!!!!!!!!!!!!!!!!!!!

graph_path = 'drive/MyDrive/Beeline Banner AB-test/graph'
data_path = 'drive/MyDrive/Beeline Banner AB-test/data'

Mounted at /content/drive


### Загружаем данные на Google Disk

In [2]:
supertable = pd.read_csv(f'{data_path}/supertable.csv')
supertable

,user_id,timestamp,date,shows,clicks,banner_first,campaign_first,placement,device_type,os,geo,has_install_event,has_first_order_event,has_registration_event,has_tariff_switch_event,segment,tariff,creative_type,size,target_audience_segment
0,1,2025-02-10 10:06:23,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
1,1,2025-02-15 20:05:59,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
2,1,2025-02-16 08:34:35,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
3,1,2025-02-17 21:55:14,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
4,1,2025-03-01 20:13:19,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,premium,premium,no_impression,no_impression,no_impression
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1171843,99999,2025-03-02 10:11:07,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,value,plus,no_impression,no_impression,no_impression
1171844,99999,2025-03-05 08:27:40,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,value,plus,no_impression,no_impression,no_impression
1171845,99999,2025-03-07 02:11:51,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,value,plus,no_impression,no_impression,no_impression
1171846,99999,2025-03-07 23:34:46,NaN,0,0,NaN,no_impression,NaN,NaN,NaN,NaN,0,0,0,0,value,plus,no_impression,no_impression,no_impression


### Исследование OS

In [44]:
# Разрез по os
os_stats = (
    supertable
    .groupby("os")
    .agg(
        users=("user_id", "nunique"),
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_install=('user_id', lambda x: supertable.loc[x.index].query('has_install_event == 1')['user_id'].nunique()),
        has_registration=('user_id', lambda x: supertable.loc[x.index].query('has_registration_event == 1')['user_id'].nunique()),
        has_first_order	= ('user_id', lambda x: supertable.loc[x.index].query('has_first_order_event == 1')['user_id'].nunique()),
        has_tariff_switch = ('user_id', lambda x: supertable.loc[x.index].query('has_tariff_switch_event == 1')['user_id'].nunique()),
    )
    .reset_index()
)

os_stats["ctr"] = os_stats["clicks"] / os_stats["shows"]
#os_stats["cr_sh_inst"] = os_stats["has_install"] / os_stats["clicks"]
os_stats["cr_inst_reg"] = os_stats["has_registration"] / os_stats["has_install"]
os_stats["cr_reg_ord"] = os_stats["has_first_order"] / os_stats["has_registration"]
os_stats["cr_reg_sw"] = os_stats["has_tariff_switch"] / os_stats["has_registration"]
print("\n OS метрики")
print(os_stats.sort_values("shows", ascending=False).head())


 OS метрики
        os  users  shows  clicks  has_install  has_registration  \
1      ios    670    920      75          397               366   
0  android    639    763      67          387               359   
2      web    130    132      15           77                73   

   has_first_order  has_tariff_switch       ctr  cr_inst_reg  cr_reg_ord  \
1              277                 97  0.081522     0.921914    0.756831   
0              243                105  0.087811     0.927649    0.676880   
2               40                 17  0.113636     0.948052    0.547945   

   cr_reg_sw  
1   0.265027  
0   0.292479  
2   0.232877  


In [22]:
ctr_mob = (os_stats["clicks"][0]+ os_stats["clicks"][1])/ (os_stats["shows"][0]+ os_stats["shows"][1])
ctr_web = (os_stats["clicks"][2] / os_stats["shows"][2])
ctr_web / ctr_mob

np.float64(1.346830985915493)

Можно выдвинуть гипотезу, для пользователей web ctr на 30% выше, чем для пользователей mob

### Исследование creative_type

In [24]:
# Разрез по creative_type
creative_stats = (
    supertable
    .groupby("creative_type")
    .agg(
        users=("user_id", "nunique"),
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_install=('user_id', lambda x: supertable.loc[x.index].query('has_install_event == 1')['user_id'].nunique()),
        has_registration=('user_id', lambda x: supertable.loc[x.index].query('has_registration_event == 1')['user_id'].nunique()),
        has_first_order	= ('user_id', lambda x: supertable.loc[x.index].query('has_first_order_event == 1')['user_id'].nunique()),
        has_tariff_switch = ('user_id', lambda x: supertable.loc[x.index].query('has_tariff_switch_event == 1')['user_id'].nunique()),
    )
    .reset_index()
)

creative_stats["ctr"] = creative_stats["clicks"] / creative_stats["shows"]
#creative_stats["cr_sh_inst"] = creative_stats["has_install"] / creative_stats["clicks"]
creative_stats["cr_inst_reg"] = creative_stats["has_registration"] / creative_stats["has_install"]
creative_stats["cr_reg_ord"] = creative_stats["has_first_order"] / creative_stats["has_registration"]
creative_stats["cr_reg_sw"] = creative_stats["has_tariff_switch"] / creative_stats["has_registration"]
print("\n OS метрики")
print(creative_stats.sort_values("shows", ascending=False).head())


 OS метрики
   creative_type   users  shows  clicks  has_install  has_registration  \
2         static     767    974      76          463               426   
3          video     566    717      71          331               313   
1           rich     104    124      10           66                58   
0  no_impression  100000      0       0         2433              2277   

   has_first_order  has_tariff_switch       ctr  cr_inst_reg  cr_reg_ord  \
2              300                118  0.078029     0.920086    0.704225   
3              223                 89  0.099024     0.945619    0.712460   
1               35                 13  0.080645     0.878788    0.603448   
0             1966               1020       NaN     0.935882    0.863417   

   cr_reg_sw  
2   0.276995  
3   0.284345  
1   0.224138  
0   0.447958  


In [27]:
creative_stats['ctr'][3] / creative_stats['ctr'][2]

np.float64(1.2690670190119648)

Анимированные баннеры дают не менее чем на 25% выше CTR

### Исследование device

In [31]:
# Разрез по device_type
device_stats = (
    supertable
    .groupby("device_type")
    .agg(
        users=("user_id", "nunique"),
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_registration=('user_id', lambda x: supertable.loc[x.index].query('has_registration_event == 1')['user_id'].nunique()),
        has_install=('user_id', lambda x: supertable.loc[x.index].query('has_install_event == 1')['user_id'].nunique()),
        has_first_order	= ('user_id', lambda x: supertable.loc[x.index].query('has_first_order_event == 1')['user_id'].nunique()),
        has_tariff_switch = ('user_id', lambda x: supertable.loc[x.index].query('has_tariff_switch_event == 1')['user_id'].nunique()),
    )
    .reset_index()
)

device_stats["ctr"] = device_stats["clicks"] / device_stats["shows"]
#device_stats["cr_sh_inst"] = device_stats["has_install"] / device_stats["clicks"]
device_stats["cr_inst_reg"] = device_stats["has_registration"] / device_stats["has_install"]
device_stats["cr_reg_ord"] = device_stats["has_first_order"] / device_stats["has_registration"]
device_stats["cr_reg_sw"] = device_stats["has_tariff_switch"] / device_stats["has_registration"]
print("\n device метрики")
print(device_stats.sort_values("shows", ascending=False).head())


 device метрики
  device_type  users  shows  clicks  has_registration  has_install  \
0       phone   1142   1529     137               671          726   
1      tablet    243    286      20               123          131   

   has_first_order  has_tariff_switch       ctr  cr_inst_reg  cr_reg_ord  \
0              470                182  0.089601     0.924242    0.700447   
1               89                 37  0.069930     0.938931    0.723577   

   cr_reg_sw  
0   0.271237  
1   0.300813  


In [33]:
device_stats["cr_reg_sw"][1] / device_stats["cr_reg_sw"][0]

np.float64(1.1090413651389261)

Реклама на tablet на 10% больше приводит к смене тарифа после регистрации

### Исследование placement

In [36]:
# Разрез по placement
placement_stats = (
    supertable
    .groupby("placement")
    .agg(
        users=("user_id", "nunique"),
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_install=('user_id', lambda x: supertable.loc[x.index].query('has_install_event == 1')['user_id'].nunique()),
        has_registration=('user_id', lambda x: supertable.loc[x.index].query('has_registration_event == 1')['user_id'].nunique()),
        has_first_order	= ('user_id', lambda x: supertable.loc[x.index].query('has_first_order_event == 1')['user_id'].nunique()),
        has_tariff_switch = ('user_id', lambda x: supertable.loc[x.index].query('has_tariff_switch_event == 1')['user_id'].nunique()),
    )
    .reset_index()
)

placement_stats["ctr"] = placement_stats["clicks"] / placement_stats["shows"]
#placement_stats["cr_sh_inst"] = placement_stats["has_install"] / placement_stats["clicks"]
placement_stats["cr_inst_reg"] = placement_stats["has_registration"] / placement_stats["has_install"]
placement_stats["cr_reg_ord"] = placement_stats["has_first_order"] / placement_stats["has_registration"]
placement_stats["cr_reg_sw"] = placement_stats["has_tariff_switch"] / placement_stats["has_registration"]
placement_stats["cr_all"] = placement_stats["has_tariff_switch"] / placement_stats["shows"]
print("\n placement метрики")
print(placement_stats.sort_values("shows", ascending=False).head())


 placement метрики
  placement  users  shows  clicks  has_install  has_registration  \
1      site    591    744      57          352               315   
0       app    562    679      66          331               318   
2    social    321    392      34          180               166   

   has_first_order  has_tariff_switch       ctr  cr_inst_reg  cr_reg_ord  \
1              237                 81  0.076613     0.894886    0.752381   
0              205                 96  0.097202     0.960725    0.644654   
2              120                 44  0.086735     0.922222    0.722892   

   cr_reg_sw    cr_all  
1   0.257143  0.108871  
0   0.301887  0.141384  
2   0.265060  0.112245  


In [41]:
cr_all_app = placement_stats["cr_all"][0]
cr_all_site_social = (placement_stats["has_tariff_switch"][1] + placement_stats["has_tariff_switch"][2]) / (placement_stats["shows"][1] + placement_stats["shows"][2])
cr_all_app / cr_all_site_social

np.float64(1.284901325478645)

Реклама в приложении эффективнее (по смене тарифа) на 25% рекламы в соц сетях / на сайте

### Исследование Географии

In [43]:
# Разрез по geo
geo_stats = (
    supertable
    .groupby("geo")
    .agg(
        users=("user_id", "nunique"),
        shows=("shows", "sum"),
        clicks=("clicks", "sum"),
        has_registration=('user_id', lambda x: supertable.loc[x.index].query('has_registration_event == 1')['user_id'].nunique()),
        has_install=('user_id', lambda x: supertable.loc[x.index].query('has_install_event == 1')['user_id'].nunique()),
        has_first_order	= ('user_id', lambda x: supertable.loc[x.index].query('has_first_order_event == 1')['user_id'].nunique()),
        has_tariff_switch = ('user_id', lambda x: supertable.loc[x.index].query('has_tariff_switch_event == 1')['user_id'].nunique()),
    )
    .reset_index()
)

geo_stats["ctr"] = geo_stats["clicks"] / geo_stats["shows"]
geo_stats["cr_reg_ord"] = geo_stats["has_first_order"] / geo_stats["has_registration"]
#geo_stats["cr_cl_inst"] = geo_stats["has_install"] / geo_stats["clicks"]
geo_stats["cr_reg_sw"] = geo_stats["has_tariff_switch"] / geo_stats["has_registration"]
geo_stats["cr_all"] = geo_stats["has_tariff_switch"] / geo_stats["shows"]
print("\n geo метрики")
print(geo_stats.sort_values("shows", ascending=False).head())



 geo метрики
               geo  users  shows  clicks  has_registration  has_install  \
2           Москва    436    522      52               240          259   
8  Санкт-Петербург    195    223      15                99          103   
5             Омск    145    181      17                77           82   
6   Ростов-на-Дону    153    167      16                77           82   
4      Новосибирск    145    164      12                76           77   

   has_first_order  has_tariff_switch       ctr  cr_reg_ord  cr_cl_inst  \
2              156                 68  0.099617    0.650000    4.980769   
8               75                 27  0.067265    0.757576    6.866667   
5               59                 17  0.093923    0.766234    4.823529   
6               60                 21  0.095808    0.779221    5.125000   
4               47                 23  0.073171    0.618421    6.416667   

   cr_reg_sw    cr_all  
2   0.283333  0.130268  
8   0.272727  0.121076  
5   0.220